# A3.7 · The unmanaged agent problem

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

---

**Risk.** Personal, non-sandboxed agents on managed endpoints.

**Control.** Endpoint + secure web gateway, behavioural monitoring — with an honest account of the gap.

**This lab.** Detect an unmanaged personal agent on a managed endpoint.

| | |
|---|---|
| Open-source tooling | Falco, osquery |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A3.7"))

The unmanaged agent problem: the agents you know about are not the ones that will hurt you. Discovery has to run against behaviour, because registration is voluntary and voluntary means partial.

In [ ]:
from cybercommons import soc, grc
import time

now = time.time()
# telemetry from three actors — no registry, just behaviour
events  = [soc.Event(now + i * 0.08, "svc-ci-runner", "read_file") for i in range(80)]
events += [soc.Event(now + t, "dana", "read_file")
           for t in (0, 6, 9, 45, 91, 140, 260, 420)]
events += [soc.Event(now + i * 0.5, "unknown-token-7f3", "http_get") for i in range(40)]

for actor in ("svc-ci-runner", "dana", "unknown-token-7f3"):
    r = soc.agent_score(events, actor)
    print(f"{actor:20s} score={r['score']:.3f}  {r['verdict']:8s} {r['signals']}")

Two of these behave like software. Only one is in anybody's inventory. That third row is what "shadow AI" looks like in telemetry before it has a name.

In [ ]:
found = grc.AIAsset("unknown-token-7f3", "agent", owner="", autonomy="L2.5",
                    data=("customer",), shadow=True)
print(grc.risk_tier(found))
for g in found.gaps():
    print("  ⚠", g)

### Expect

`svc-ci-runner` and `unknown-token-7f3` score as agents; `dana` scores as human. The discovered agent tiers high or critical and reports gaps for missing ownership and registration.

### Your turn

Run `agent_score` over a day of real authentication logs. Every actor scoring above 0.6 that is not in your NHI inventory is a finding — and the first run always produces some.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A3.7.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*